<a href="https://colab.research.google.com/github/amritapathak1/30100_project_FromDiscussiontoDecision/blob/main/redditdata30100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/content

/content/drive/MyDrive/content


In [4]:
!pip install zstandard

In [5]:
import zstandard
import os
import json
import sys
from datetime import datetime

def read_and_decode(reader, chunk_size, max_window_size, previous_chunk=None, bytes_read=0):
	chunk = reader.read(chunk_size)
	bytes_read += chunk_size
	if previous_chunk is not None:
		chunk = previous_chunk + chunk
	try:
		return chunk.decode()
	except UnicodeDecodeError:
		if bytes_read > max_window_size:
			raise UnicodeError(f"Unable to decode frame after reading {bytes_read:,} bytes")
		log.info(f"Decoding error with {bytes_read:,} bytes, reading another chunk")
		return read_and_decode(reader, chunk_size, max_window_size, chunk, bytes_read)


def read_lines_zst(file_name):
	with open(file_name, 'rb') as file_handle:
		buffer = ''
		reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(file_handle)
		while True:
			chunk = read_and_decode(reader, 2**27, (2**29) * 2)

			if not chunk:
				break
			lines = (buffer + chunk).split("\n")

			for line in lines[:-1]:
				yield line, file_handle.tell()

			buffer = lines[-1]

		reader.close()

In [ ]:

file_size = os.path.getsize('RS_2015-05.zst')
file_size

1686563661

In [ ]:

file_size = os.path.getsize('RC_2015-05.zst')
file_size

5459532839

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-05.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-05_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-05.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-05-01 15:07:56 : 100,000 : 62 : 42,730,450 : 3%
2015-05-02 02:41:27 : 200,000 : 150 : 84,805,525 : 5%
2015-05-02 19:05:28 : 300,000 : 211 : 106,301,825 : 6%
2015-05-03 12:01:27 : 400,000 : 265 : 128,715,650 : 8%
2015-05-04 00:38:15 : 500,000 : 312 : 174,067,600 : 10%
2015-05-04 15:43:42 : 600,000 : 377 : 196,743,575 : 12%
2015-05-05 02:03:15 : 700,000 : 453 : 219,550,625 : 13%
2015-05-05 16:42:20 : 800,000 : 554 : 264,771,500 : 16%
2015-05-06 03:15:53 : 900,000 : 659 : 287,316,400 : 17%
2015-05-06 17:33:18 : 1,000,000 : 733 : 309,992,375 : 18%
2015-05-07 04:38:47 : 1,100,000 : 841 : 332,537,275 : 20%
2015-05-07 18:43:00 : 1,200,000 : 942 : 377,627,075 : 22%
2015-05-08 06:39:32 : 1,300,000 : 1,000 : 400,040,900 : 24%
2015-05-08 20:24:02 : 1,400,000 : 1,086 : 422,585,800 : 25%
2015-05-09 12:44:20 : 1,500,000 : 1,154 : 467,544,525 : 28%
2015-05-10 01:40:26 : 1,600,000 : 1,209 : 489,958,350 : 29%
2015-05-10 18:28:09 : 1,700,000 : 1,268 : 512,372,175 : 30%
2015-05-11 07:57:49 : 1,800,0

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-05.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-05_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-05.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-05-01 00:57:33 : 100,000 : 0 : 20,972,000 : 0%
2015-05-01 01:55:43 : 200,000 : 0 : 20,972,000 : 0%
2015-05-01 02:54:57 : 300,000 : 0 : 42,206,150 : 1%
2015-05-01 04:02:35 : 400,000 : 0 : 42,206,150 : 1%
2015-05-01 05:31:14 : 500,000 : 0 : 64,095,675 : 1%
2015-05-01 07:39:37 : 600,000 : 0 : 64,095,675 : 1%
2015-05-01 10:32:22 : 700,000 : 0 : 85,723,050 : 2%
2015-05-01 12:39:38 : 800,000 : 0 : 85,723,050 : 2%
2015-05-01 14:01:18 : 900,000 : 0 : 108,136,875 : 2%
2015-05-01 15:06:34 : 1,000,000 : 0 : 108,136,875 : 2%
2015-05-01 16:05:56 : 1,100,000 : 0 : 108,136,875 : 2%
2015-05-01 17:03:46 : 1,200,000 : 0 : 130,681,775 : 2%
2015-05-01 18:02:00 : 1,300,000 : 0 : 130,681,775 : 2%
2015-05-01 19:00:51 : 1,400,000 : 0 : 152,833,450 : 3%
2015-05-01 20:00:58 : 1,500,000 : 0 : 152,833,450 : 3%
2015-05-01 21:03:22 : 1,600,000 : 0 : 174,985,125 : 3%
2015-05-01 22:11:36 : 1,700,000 : 0 : 174,985,125 : 3%
2015-05-01 23:24:52 : 1,800,000 : 0 : 196,219,275 : 4%
2015-05-02 00:36:43 : 1,900,000 : 0 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-06.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-06_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-06.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-06-01 13:43:16 : 100,000 : 74 : 43,647,975 : 2%
2015-06-01 22:40:56 : 200,000 : 184 : 65,668,575 : 4%
2015-06-02 11:14:35 : 300,000 : 248 : 109,185,475 : 6%
2015-06-02 19:55:16 : 400,000 : 317 : 130,419,625 : 7%
2015-06-03 04:59:35 : 500,000 : 395 : 172,887,925 : 9%
2015-06-03 16:29:40 : 600,000 : 457 : 193,859,925 : 10%
2015-06-04 00:29:43 : 700,000 : 535 : 236,197,150 : 13%
2015-06-04 12:58:23 : 800,000 : 587 : 257,038,075 : 14%
2015-06-04 21:03:33 : 900,000 : 659 : 299,113,150 : 16%
2015-06-05 06:33:12 : 1,000,000 : 710 : 339,615,325 : 18%
2015-06-05 17:57:40 : 1,100,000 : 775 : 360,456,250 : 19%
2015-06-06 04:00:43 : 1,200,000 : 841 : 402,269,175 : 22%
2015-06-06 18:33:30 : 1,300,000 : 902 : 422,979,025 : 23%
2015-06-07 06:54:53 : 1,400,000 : 966 : 464,529,800 : 25%
2015-06-07 20:44:25 : 1,500,000 : 1,016 : 506,735,950 : 27%
2015-06-08 09:03:00 : 1,600,000 : 1,079 : 548,811,025 : 29%
2015-06-08 20:03:23 : 1,700,000 : 1,138 : 570,307,325 : 31%
2015-06-09 06:39:00 : 1,800,000 : 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-07.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-07_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-07.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-07-01 14:37:13 : 100,000 : 75 : 44,041,200 : 2%
2015-07-02 00:29:48 : 200,000 : 175 : 65,930,725 : 4%
2015-07-02 15:07:19 : 300,000 : 243 : 109,971,925 : 6%
2015-07-03 01:40:29 : 400,000 : 336 : 131,861,450 : 7%
2015-07-03 15:45:50 : 500,000 : 412 : 153,488,825 : 9%
2015-07-04 02:53:23 : 600,000 : 466 : 196,481,425 : 11%
2015-07-04 18:45:08 : 700,000 : 522 : 217,846,650 : 12%
2015-07-05 11:23:57 : 800,000 : 596 : 239,605,100 : 13%
2015-07-06 00:50:31 : 900,000 : 667 : 283,122,000 : 16%
2015-07-06 15:37:14 : 1,000,000 : 744 : 305,142,600 : 17%
2015-07-07 01:36:20 : 1,100,000 : 853 : 349,577,025 : 19%
2015-07-07 15:35:02 : 1,200,000 : 932 : 371,597,625 : 21%
2015-07-08 01:10:59 : 1,300,000 : 1,021 : 393,487,150 : 22%
2015-07-08 15:12:57 : 1,400,000 : 1,115 : 437,528,350 : 24%
2015-07-09 00:49:20 : 1,500,000 : 1,201 : 459,811,100 : 26%
2015-07-09 14:46:17 : 1,600,000 : 1,277 : 481,831,700 : 27%
2015-07-10 00:43:33 : 1,700,000 : 1,362 : 525,872,900 : 29%
2015-07-10 15:00:15 : 1,800,00

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-08.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-08_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-08.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-08-01 15:50:25 : 100,000 : 49 : 43,385,825 : 2%
2015-08-02 03:57:12 : 200,000 : 104 : 65,013,200 : 3%
2015-08-02 19:48:59 : 300,000 : 161 : 108,267,950 : 6%
2015-08-03 09:00:01 : 400,000 : 245 : 130,026,400 : 7%
2015-08-03 20:23:16 : 500,000 : 316 : 151,784,850 : 8%
2015-08-04 07:51:47 : 600,000 : 400 : 195,301,750 : 10%
2015-08-04 19:35:37 : 700,000 : 457 : 217,060,200 : 12%
2015-08-05 06:22:58 : 800,000 : 531 : 260,183,875 : 14%
2015-08-05 18:41:08 : 900,000 : 642 : 282,204,475 : 15%
2015-08-06 04:56:43 : 1,000,000 : 726 : 325,721,375 : 17%
2015-08-06 17:47:26 : 1,100,000 : 799 : 347,348,750 : 19%
2015-08-07 03:54:12 : 1,200,000 : 909 : 369,238,275 : 20%
2015-08-07 17:37:36 : 1,300,000 : 981 : 412,624,100 : 22%
2015-08-08 04:43:10 : 1,400,000 : 1,078 : 434,251,475 : 23%
2015-08-08 19:47:32 : 1,500,000 : 1,130 : 476,981,925 : 25%
2015-08-09 10:31:44 : 1,600,000 : 1,177 : 498,478,225 : 27%
2015-08-09 23:16:53 : 1,700,000 : 1,249 : 520,105,600 : 28%
2015-08-10 13:17:10 : 1,800,000 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-09.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-09_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-09.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-09-01 12:53:42 : 100,000 : 73 : 40,371,100 : 2%
2015-09-01 22:12:19 : 200,000 : 159 : 61,212,025 : 4%
2015-09-02 10:21:25 : 300,000 : 226 : 101,452,050 : 6%
2015-09-02 20:30:24 : 400,000 : 299 : 122,292,975 : 7%
2015-09-03 08:26:53 : 500,000 : 387 : 142,478,525 : 8%
2015-09-03 19:57:56 : 600,000 : 484 : 183,636,075 : 11%
2015-09-04 07:25:33 : 700,000 : 566 : 203,559,475 : 12%
2015-09-04 19:02:54 : 800,000 : 632 : 244,454,875 : 14%
2015-09-05 06:37:55 : 900,000 : 702 : 264,378,275 : 15%
2015-09-05 20:04:11 : 1,000,000 : 755 : 284,825,975 : 16%
2015-09-06 10:10:03 : 1,100,000 : 805 : 325,328,150 : 19%
2015-09-06 22:47:44 : 1,200,000 : 865 : 345,775,850 : 20%
2015-09-07 13:16:33 : 1,300,000 : 935 : 386,278,025 : 22%
2015-09-08 00:20:06 : 1,400,000 : 1,017 : 406,332,500 : 23%
2015-09-08 14:01:50 : 1,500,000 : 1,089 : 426,255,900 : 24%
2015-09-08 23:15:33 : 1,600,000 : 1,178 : 466,758,075 : 27%
2015-09-09 09:38:53 : 1,700,000 : 1,235 : 484,846,425 : 28%
2015-09-09 19:06:06 : 1,800,000 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-10.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-10_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-10.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-10-01 13:14:33 : 100,000 : 72 : 40,895,400 : 2%
2015-10-01 22:51:45 : 200,000 : 191 : 61,605,250 : 3%
2015-10-02 13:24:16 : 300,000 : 272 : 102,762,800 : 6%
2015-10-02 23:30:42 : 400,000 : 352 : 123,603,725 : 7%
2015-10-03 15:19:02 : 500,000 : 408 : 143,920,350 : 8%
2015-10-04 02:48:12 : 600,000 : 474 : 185,077,900 : 10%
2015-10-04 18:15:04 : 700,000 : 531 : 205,525,600 : 11%
2015-10-05 04:59:06 : 800,000 : 581 : 246,158,850 : 14%
2015-10-05 18:08:54 : 900,000 : 677 : 267,130,850 : 15%
2015-10-06 04:00:54 : 1,000,000 : 809 : 287,840,700 : 16%
2015-10-06 16:59:58 : 1,100,000 : 933 : 328,998,250 : 18%
2015-10-07 02:28:20 : 1,200,000 : 1,038 : 349,577,025 : 19%
2015-10-07 15:46:16 : 1,300,000 : 1,152 : 390,865,650 : 22%
2015-10-08 01:21:17 : 1,400,000 : 1,232 : 411,444,425 : 23%
2015-10-08 14:48:20 : 1,500,000 : 1,317 : 431,629,975 : 24%
2015-10-09 00:20:49 : 1,600,000 : 1,421 : 472,918,600 : 26%
2015-10-09 14:16:08 : 1,700,000 : 1,499 : 493,235,225 : 27%
2015-10-10 00:30:55 : 1,800,

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-11.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-11_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-11.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-11-01 16:03:33 : 100,000 : 60 : 41,288,625 : 2%
2015-11-02 01:40:43 : 200,000 : 115 : 61,867,400 : 4%
2015-11-02 15:25:38 : 300,000 : 175 : 103,418,175 : 6%
2015-11-02 23:58:00 : 400,000 : 248 : 124,259,100 : 7%
2015-11-03 12:34:06 : 500,000 : 312 : 144,837,875 : 8%
2015-11-03 21:45:39 : 600,000 : 423 : 186,388,650 : 11%
2015-11-04 08:47:43 : 700,000 : 504 : 206,967,425 : 12%
2015-11-04 20:00:39 : 800,000 : 587 : 227,808,350 : 13%
2015-11-05 06:05:31 : 900,000 : 695 : 269,359,125 : 16%
2015-11-05 18:39:21 : 1,000,000 : 791 : 290,068,975 : 17%
2015-11-06 04:20:36 : 1,100,000 : 871 : 331,488,675 : 19%
2015-11-06 17:36:16 : 1,200,000 : 949 : 352,198,525 : 20%
2015-11-07 03:37:30 : 1,300,000 : 1,022 : 372,646,225 : 22%
2015-11-07 18:31:57 : 1,400,000 : 1,089 : 413,672,700 : 24%
2015-11-08 05:55:17 : 1,500,000 : 1,150 : 434,120,400 : 25%
2015-11-08 20:20:41 : 1,600,000 : 1,213 : 475,146,875 : 28%
2015-11-09 07:14:05 : 1,700,000 : 1,283 : 495,725,650 : 29%
2015-11-09 19:24:10 : 1,800,00

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2015-12.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2015-12_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2015-12.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-12-01 12:13:13 : 100,000 : 57 : 41,812,925 : 2%
2015-12-01 21:27:19 : 200,000 : 133 : 62,653,850 : 4%
2015-12-02 07:32:13 : 300,000 : 217 : 103,549,250 : 6%
2015-12-02 18:59:43 : 400,000 : 312 : 124,259,100 : 7%
2015-12-03 04:26:24 : 500,000 : 423 : 144,968,950 : 8%
2015-12-03 17:31:23 : 600,000 : 508 : 186,126,500 : 11%
2015-12-04 02:43:09 : 700,000 : 625 : 206,705,275 : 12%
2015-12-04 15:58:39 : 800,000 : 705 : 227,152,975 : 13%
2015-12-05 01:28:17 : 900,000 : 808 : 268,179,450 : 15%
2015-12-05 16:26:23 : 1,000,000 : 874 : 288,102,850 : 16%
2015-12-06 02:49:22 : 1,100,000 : 930 : 308,288,400 : 18%
2015-12-06 17:42:04 : 1,200,000 : 988 : 348,659,500 : 20%
2015-12-07 03:26:46 : 1,300,000 : 1,072 : 369,238,275 : 21%
2015-12-07 16:38:52 : 1,400,000 : 1,169 : 389,817,050 : 22%
2015-12-08 01:27:46 : 1,500,000 : 1,279 : 431,105,675 : 25%
2015-12-08 14:20:54 : 1,600,000 : 1,358 : 451,291,225 : 26%
2015-12-08 22:48:24 : 1,700,000 : 1,461 : 491,006,950 : 28%
2015-12-09 10:05:11 : 1,800,00

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2016-01.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2016-01_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2016-01.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-01-01 17:02:44 : 100,000 : 47 : 40,502,175 : 2%
2016-01-02 04:03:56 : 200,000 : 99 : 60,818,800 : 3%
2016-01-02 18:33:26 : 300,000 : 138 : 100,796,675 : 5%
2016-01-03 05:01:33 : 400,000 : 197 : 120,982,225 : 6%
2016-01-03 19:07:13 : 500,000 : 246 : 141,036,700 : 7%
2016-01-04 04:58:01 : 600,000 : 292 : 180,621,350 : 9%
2016-01-04 17:43:28 : 700,000 : 355 : 201,200,125 : 10%
2016-01-05 02:40:46 : 800,000 : 417 : 242,488,750 : 13%
2016-01-05 15:11:06 : 900,000 : 475 : 263,067,525 : 14%
2016-01-05 23:37:43 : 1,000,000 : 550 : 283,646,300 : 15%
2016-01-06 11:12:04 : 1,100,000 : 602 : 324,410,625 : 17%
2016-01-06 20:58:14 : 1,200,000 : 703 : 345,120,475 : 18%
2016-01-07 06:20:50 : 1,300,000 : 770 : 365,568,175 : 19%
2016-01-07 18:21:52 : 1,400,000 : 831 : 406,594,650 : 21%
2016-01-08 02:59:11 : 1,500,000 : 902 : 427,435,575 : 22%
2016-01-08 15:50:58 : 1,600,000 : 977 : 447,752,200 : 23%
2016-01-09 00:37:39 : 1,700,000 : 1,062 : 488,909,750 : 25%
2016-01-09 14:43:36 : 1,800,000 : 1,110 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2016-02.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2016-02_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2016-02.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-02-01 11:26:16 : 100,000 : 67 : 40,633,250 : 2%
2016-02-01 20:18:49 : 200,000 : 129 : 60,949,875 : 3%
2016-02-02 04:53:16 : 300,000 : 209 : 100,927,750 : 5%
2016-02-02 16:46:45 : 400,000 : 284 : 121,637,600 : 6%
2016-02-03 00:52:23 : 500,000 : 357 : 142,216,375 : 7%
2016-02-03 13:10:24 : 600,000 : 420 : 182,849,625 : 10%
2016-02-03 21:41:46 : 700,000 : 490 : 203,297,325 : 11%
2016-02-04 08:22:33 : 800,000 : 563 : 244,192,725 : 13%
2016-02-04 19:11:46 : 900,000 : 640 : 264,902,575 : 14%
2016-02-05 04:09:52 : 1,000,000 : 721 : 285,219,200 : 15%
2016-02-05 16:41:19 : 1,100,000 : 786 : 325,983,525 : 17%
2016-02-06 01:21:03 : 1,200,000 : 887 : 346,300,150 : 18%
2016-02-06 14:58:24 : 1,300,000 : 935 : 365,961,400 : 19%
2016-02-07 00:13:36 : 1,400,000 : 991 : 406,201,425 : 21%
2016-02-07 12:58:44 : 1,500,000 : 1,039 : 425,338,375 : 22%
2016-02-07 22:36:30 : 1,600,000 : 1,094 : 465,709,475 : 24%
2016-02-08 11:27:42 : 1,700,000 : 1,148 : 486,157,175 : 25%
2016-02-08 20:56:55 : 1,800,000 : 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2016-03.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2016-03_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2016-03.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-03-01 10:47:03 : 100,000 : 91 : 40,502,175 : 2%
2016-03-01 19:11:38 : 200,000 : 177 : 61,080,950 : 3%
2016-03-02 03:49:42 : 300,000 : 305 : 102,107,425 : 5%
2016-03-02 16:00:29 : 400,000 : 376 : 122,817,275 : 6%
2016-03-02 23:25:33 : 500,000 : 461 : 143,133,900 : 7%
2016-03-03 09:20:57 : 600,000 : 552 : 163,450,525 : 8%
2016-03-03 19:38:30 : 700,000 : 632 : 204,608,075 : 10%
2016-03-04 04:59:21 : 800,000 : 743 : 225,186,850 : 11%
2016-03-04 17:17:52 : 900,000 : 813 : 266,475,475 : 14%
2016-03-05 02:19:14 : 1,000,000 : 987 : 286,923,175 : 15%
2016-03-05 16:30:45 : 1,100,000 : 1,047 : 307,108,725 : 16%
2016-03-06 02:27:00 : 1,200,000 : 1,125 : 347,741,975 : 18%
2016-03-06 16:29:44 : 1,300,000 : 1,171 : 367,796,450 : 19%
2016-03-07 01:55:07 : 1,400,000 : 1,232 : 408,560,775 : 21%
2016-03-07 14:22:56 : 1,500,000 : 1,286 : 429,270,625 : 22%
2016-03-07 22:27:31 : 1,600,000 : 1,400 : 449,980,475 : 23%
2016-03-08 08:28:14 : 1,700,000 : 1,479 : 491,006,950 : 25%
2016-03-08 19:09:30 : 1,800

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2016-04.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2016-04_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2016-04.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-04-01 11:15:26 : 100,000 : 79 : 42,992,600 : 2%
2016-04-01 19:08:48 : 200,000 : 164 : 64,357,825 : 3%
2016-04-02 03:28:48 : 300,000 : 272 : 85,723,050 : 4%
2016-04-02 16:52:37 : 400,000 : 330 : 128,322,425 : 7%
2016-04-03 02:43:52 : 500,000 : 396 : 149,818,725 : 8%
2016-04-03 16:56:22 : 600,000 : 483 : 171,183,950 : 9%
2016-04-04 02:21:47 : 700,000 : 550 : 192,942,400 : 10%
2016-04-04 14:54:27 : 800,000 : 622 : 236,328,225 : 12%
2016-04-04 22:58:47 : 900,000 : 757 : 258,217,750 : 13%
2016-04-05 10:49:05 : 1,000,000 : 828 : 279,714,050 : 14%
2016-04-05 19:44:18 : 1,100,000 : 923 : 323,362,025 : 17%
2016-04-06 05:03:26 : 1,200,000 : 992 : 344,858,325 : 18%
2016-04-06 16:48:09 : 1,300,000 : 1,083 : 366,878,925 : 19%
2016-04-07 01:15:36 : 1,400,000 : 1,175 : 410,657,975 : 21%
2016-04-07 13:44:42 : 1,500,000 : 1,300 : 432,285,350 : 22%
2016-04-07 21:41:00 : 1,600,000 : 1,457 : 454,174,875 : 23%
2016-04-08 08:47:22 : 1,700,000 : 1,548 : 475,802,250 : 24%
2016-04-08 18:48:03 : 1,800,000 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RS_2016-05.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RS-2016-05_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RS_2016-05.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-05-01 13:50:35 : 100,000 : 70 : 41,026,475 : 2%
2016-05-01 20:22:55 : 200,000 : 124 : 58,983,750 : 3%
2016-05-02 02:14:56 : 300,000 : 157 : 76,416,725 : 4%
2016-05-02 09:04:24 : 400,000 : 227 : 91,883,575 : 5%
2016-05-02 15:11:36 : 500,000 : 310 : 108,923,325 : 5%
2016-05-02 22:18:27 : 600,000 : 452 : 151,784,850 : 8%
2016-05-03 08:52:08 : 700,000 : 569 : 173,019,000 : 9%
2016-05-03 17:35:33 : 800,000 : 699 : 191,631,650 : 10%
2016-05-03 22:10:27 : 900,000 : 778 : 209,064,625 : 10%
2016-05-04 05:05:34 : 1,000,000 : 832 : 229,250,175 : 11%
2016-05-04 16:24:37 : 1,100,000 : 968 : 271,849,550 : 14%
2016-05-05 00:19:08 : 1,200,000 : 1,097 : 293,608,000 : 15%
2016-05-05 12:49:50 : 1,300,000 : 1,205 : 314,842,150 : 16%
2016-05-05 21:10:02 : 1,400,000 : 1,350 : 357,965,825 : 18%
2016-05-06 08:37:47 : 1,500,000 : 1,442 : 379,331,050 : 19%
2016-05-06 18:52:32 : 1,600,000 : 1,591 : 400,827,350 : 20%
2016-05-07 05:05:20 : 1,700,000 : 1,676 : 422,323,650 : 21%
2016-05-07 18:26:00 : 1,800,000 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-06.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-06_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-06.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-06-01 01:09:15 : 100,000 : 0 : 21,496,300 : 0%
2015-06-01 02:17:08 : 200,000 : 0 : 21,496,300 : 0%
2015-06-01 03:26:00 : 300,000 : 0 : 43,254,750 : 1%
2015-06-01 04:48:55 : 400,000 : 0 : 43,254,750 : 1%
2015-06-01 06:33:37 : 500,000 : 0 : 65,537,500 : 1%
2015-06-01 09:00:46 : 600,000 : 0 : 65,537,500 : 1%
2015-06-01 11:39:12 : 700,000 : 0 : 88,213,475 : 2%
2015-06-01 13:24:16 : 800,000 : 0 : 88,213,475 : 2%
2015-06-01 14:41:00 : 900,000 : 0 : 111,020,525 : 2%
2015-06-01 15:46:09 : 1,000,000 : 0 : 111,020,525 : 2%
2015-06-01 16:46:10 : 1,100,000 : 0 : 133,565,425 : 2%
2015-06-01 17:43:37 : 1,200,000 : 0 : 133,565,425 : 2%
2015-06-01 18:40:32 : 1,300,000 : 0 : 133,565,425 : 2%
2015-06-01 19:36:10 : 1,400,000 : 0 : 155,979,250 : 3%
2015-06-01 20:32:20 : 1,500,000 : 0 : 155,979,250 : 3%
2015-06-01 21:29:10 : 1,600,000 : 0 : 178,393,075 : 3%
2015-06-01 22:30:02 : 1,700,000 : 0 : 178,393,075 : 3%
2015-06-01 23:35:07 : 1,800,000 : 0 : 200,413,675 : 4%
2015-06-02 00:41:43 : 1,900,000 : 0 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-07.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-07_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-07.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-07-01 01:31:26 : 100,000 : 0 : 21,889,525 : 0%
2015-07-01 02:42:08 : 200,000 : 0 : 21,889,525 : 0%
2015-07-01 03:53:00 : 300,000 : 0 : 43,647,975 : 1%
2015-07-01 05:11:54 : 400,000 : 0 : 43,647,975 : 1%
2015-07-01 06:52:41 : 500,000 : 0 : 65,799,650 : 1%
2015-07-01 09:07:53 : 600,000 : 0 : 65,799,650 : 1%
2015-07-01 11:35:58 : 700,000 : 0 : 88,082,400 : 1%
2015-07-01 13:17:06 : 800,000 : 0 : 88,082,400 : 1%
2015-07-01 14:28:19 : 900,000 : 0 : 110,627,300 : 2%
2015-07-01 15:28:32 : 1,000,000 : 0 : 110,627,300 : 2%
2015-07-01 16:23:29 : 1,100,000 : 0 : 132,778,975 : 2%
2015-07-01 17:16:30 : 1,200,000 : 0 : 132,778,975 : 2%
2015-07-01 18:09:16 : 1,300,000 : 0 : 132,778,975 : 2%
2015-07-01 19:01:03 : 1,400,000 : 0 : 155,061,725 : 3%
2015-07-01 19:53:33 : 1,500,000 : 0 : 155,061,725 : 3%
2015-07-01 20:47:25 : 1,600,000 : 0 : 177,213,400 : 3%
2015-07-01 21:45:02 : 1,700,000 : 0 : 177,213,400 : 3%
2015-07-01 22:46:18 : 1,800,000 : 0 : 198,971,850 : 3%
2015-07-01 23:51:22 : 1,900,000 : 0 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-08.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-08_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-08.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-08-01 01:10:46 : 100,000 : 0 : 23,986,725 : 0%
2015-08-01 02:22:25 : 200,000 : 0 : 23,986,725 : 0%
2015-08-01 03:38:16 : 300,000 : 0 : 47,973,450 : 1%
2015-08-01 05:04:49 : 400,000 : 0 : 47,973,450 : 1%
2015-08-01 06:51:11 : 500,000 : 0 : 47,973,450 : 1%
2015-08-01 09:19:55 : 600,000 : 0 : 72,353,400 : 1%
2015-08-01 12:09:38 : 700,000 : 0 : 72,353,400 : 1%
2015-08-01 14:08:20 : 800,000 : 0 : 96,733,350 : 2%
2015-08-01 15:36:25 : 900,000 : 0 : 96,733,350 : 2%
2015-08-01 16:53:23 : 1,000,000 : 0 : 96,733,350 : 2%
2015-08-01 18:06:25 : 1,100,000 : 0 : 120,851,150 : 2%
2015-08-01 19:17:09 : 1,200,000 : 0 : 120,851,150 : 2%
2015-08-01 20:31:21 : 1,300,000 : 0 : 144,968,950 : 3%
2015-08-01 21:46:00 : 1,400,000 : 0 : 144,968,950 : 3%
2015-08-01 23:02:28 : 1,500,000 : 0 : 144,968,950 : 3%
2015-08-02 00:17:10 : 1,600,000 : 0 : 168,562,450 : 3%
2015-08-02 01:31:07 : 1,700,000 : 0 : 168,562,450 : 3%
2015-08-02 02:46:49 : 1,800,000 : 0 : 168,562,450 : 3%
2015-08-02 04:08:31 : 1,900,000 : 0 : 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-09.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-09_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-09.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-09-01 01:00:21 : 100,000 : 0 : 23,855,650 : 0%
2015-09-01 02:00:43 : 200,000 : 0 : 23,855,650 : 0%
2015-09-01 03:01:16 : 300,000 : 0 : 47,580,225 : 1%
2015-09-01 04:10:00 : 400,000 : 0 : 47,580,225 : 1%
2015-09-01 05:35:53 : 500,000 : 0 : 47,580,225 : 1%
2015-09-01 07:30:24 : 600,000 : 0 : 71,698,025 : 1%
2015-09-01 10:04:44 : 700,000 : 0 : 71,698,025 : 1%
2015-09-01 12:13:44 : 800,000 : 0 : 96,209,050 : 2%
2015-09-01 13:59:52 : 900,000 : 0 : 96,209,050 : 2%
2015-09-01 15:08:51 : 1,000,000 : 0 : 96,209,050 : 2%
2015-09-01 16:05:12 : 1,100,000 : 0 : 120,720,075 : 2%
2015-09-01 16:59:15 : 1,200,000 : 0 : 120,720,075 : 2%
2015-09-01 17:53:38 : 1,300,000 : 0 : 145,231,100 : 3%
2015-09-01 18:48:31 : 1,400,000 : 0 : 145,231,100 : 3%
2015-09-01 19:43:30 : 1,500,000 : 0 : 145,231,100 : 3%
2015-09-01 20:38:04 : 1,600,000 : 0 : 169,479,975 : 3%
2015-09-01 21:34:02 : 1,700,000 : 0 : 169,479,975 : 3%
2015-09-01 22:33:17 : 1,800,000 : 0 : 193,335,625 : 4%
2015-09-01 23:53:50 : 1,900,000 : 0 : 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-10.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-10_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-10.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-10-01 01:01:20 : 100,000 : 0 : 23,855,650 : 0%
2015-10-01 02:03:52 : 200,000 : 0 : 23,855,650 : 0%
2015-10-01 03:08:45 : 300,000 : 0 : 47,973,450 : 1%
2015-10-01 04:21:33 : 400,000 : 0 : 47,973,450 : 1%
2015-10-01 05:54:22 : 500,000 : 0 : 47,973,450 : 1%
2015-10-01 08:01:42 : 600,000 : 0 : 72,484,475 : 1%
2015-10-01 10:44:47 : 700,000 : 0 : 72,484,475 : 1%
2015-10-01 12:50:55 : 800,000 : 0 : 96,864,425 : 2%
2015-10-01 14:04:36 : 900,000 : 0 : 96,864,425 : 2%
2015-10-01 15:04:43 : 1,000,000 : 0 : 96,864,425 : 2%
2015-10-01 15:57:38 : 1,100,000 : 0 : 121,113,300 : 2%
2015-10-01 16:49:17 : 1,200,000 : 0 : 121,113,300 : 2%
2015-10-01 17:42:19 : 1,300,000 : 0 : 145,362,175 : 3%
2015-10-01 18:35:18 : 1,400,000 : 0 : 145,362,175 : 3%
2015-10-01 19:26:35 : 1,500,000 : 0 : 145,362,175 : 3%
2015-10-01 20:18:04 : 1,600,000 : 0 : 169,479,975 : 3%
2015-10-01 21:09:55 : 1,700,000 : 0 : 169,479,975 : 3%
2015-10-01 22:07:09 : 1,800,000 : 0 : 193,335,625 : 3%
2015-10-01 23:09:53 : 1,900,000 : 0 : 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-11.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-11_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-11.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-11-01 01:17:05 : 100,000 : 0 : 22,807,050 : 0%
2015-11-01 02:30:01 : 200,000 : 0 : 22,807,050 : 0%
2015-11-01 03:47:09 : 300,000 : 0 : 46,138,400 : 1%
2015-11-01 05:26:18 : 400,000 : 0 : 46,138,400 : 1%
2015-11-01 07:40:45 : 500,000 : 0 : 46,138,400 : 1%
2015-11-01 10:42:03 : 600,000 : 0 : 69,994,050 : 1%
2015-11-01 13:16:40 : 700,000 : 0 : 69,994,050 : 1%
2015-11-01 14:55:28 : 800,000 : 0 : 93,456,475 : 2%
2015-11-01 16:09:13 : 900,000 : 0 : 93,456,475 : 2%
2015-11-01 17:14:48 : 1,000,000 : 0 : 93,456,475 : 2%
2015-11-01 18:17:13 : 1,100,000 : 0 : 116,394,600 : 2%
2015-11-01 19:12:06 : 1,200,000 : 0 : 116,394,600 : 2%
2015-11-01 20:06:47 : 1,300,000 : 0 : 116,394,600 : 2%
2015-11-01 21:00:30 : 1,400,000 : 0 : 139,201,650 : 3%
2015-11-01 21:53:07 : 1,500,000 : 0 : 139,201,650 : 3%
2015-11-01 22:50:11 : 1,600,000 : 0 : 162,401,925 : 3%
2015-11-01 23:48:47 : 1,700,000 : 0 : 162,401,925 : 3%
2015-11-02 00:48:07 : 1,800,000 : 0 : 162,401,925 : 3%
2015-11-02 01:47:50 : 1,900,000 : 0 : 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2015-12.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2015-12_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2015-12.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2015-12-01 01:01:34 : 100,000 : 0 : 23,462,425 : 0%
2015-12-01 02:00:48 : 200,000 : 0 : 23,462,425 : 0%
2015-12-01 02:59:23 : 300,000 : 0 : 45,745,175 : 1%
2015-12-01 03:58:21 : 400,000 : 0 : 45,745,175 : 1%
2015-12-01 05:01:20 : 500,000 : 0 : 68,683,300 : 1%
2015-12-01 06:22:24 : 600,000 : 0 : 68,683,300 : 1%
2015-12-01 08:15:06 : 700,000 : 0 : 68,683,300 : 1%
2015-12-01 10:49:55 : 800,000 : 0 : 92,932,175 : 2%
2015-12-01 13:05:43 : 900,000 : 0 : 92,932,175 : 2%
2015-12-01 14:33:12 : 1,000,000 : 0 : 116,918,900 : 2%
2015-12-01 15:39:15 : 1,100,000 : 0 : 116,918,900 : 2%
2015-12-01 16:37:04 : 1,200,000 : 0 : 116,918,900 : 2%
2015-12-01 17:30:14 : 1,300,000 : 0 : 140,774,550 : 3%
2015-12-01 18:21:36 : 1,400,000 : 0 : 140,774,550 : 3%
2015-12-01 19:11:27 : 1,500,000 : 0 : 164,630,200 : 3%
2015-12-01 20:00:52 : 1,600,000 : 0 : 164,630,200 : 3%
2015-12-01 20:49:04 : 1,700,000 : 0 : 188,485,850 : 3%
2015-12-01 21:37:35 : 1,800,000 : 0 : 188,485,850 : 3%
2015-12-01 22:27:51 : 1,900,000 : 0 :

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2016-01.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2016-01_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2016-01.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-01-01 01:09:26 : 100,000 : 0 : 22,544,900 : 0%
2016-01-01 02:24:37 : 200,000 : 0 : 22,544,900 : 0%
2016-01-01 03:45:45 : 300,000 : 0 : 44,827,650 : 1%
2016-01-01 05:15:03 : 400,000 : 0 : 44,827,650 : 1%
2016-01-01 07:04:29 : 500,000 : 0 : 44,827,650 : 1%
2016-01-01 09:27:42 : 600,000 : 0 : 68,027,925 : 1%
2016-01-01 12:35:24 : 700,000 : 0 : 68,027,925 : 1%
2016-01-01 15:04:12 : 800,000 : 0 : 91,097,125 : 2%
2016-01-01 16:42:19 : 900,000 : 0 : 91,097,125 : 2%
2016-01-01 17:58:34 : 1,000,000 : 0 : 91,097,125 : 2%
2016-01-01 19:02:31 : 1,100,000 : 0 : 113,642,025 : 2%
2016-01-01 20:06:17 : 1,200,000 : 0 : 113,642,025 : 2%
2016-01-01 21:10:15 : 1,300,000 : 0 : 136,449,075 : 2%
2016-01-01 22:16:04 : 1,400,000 : 0 : 136,449,075 : 2%
2016-01-01 23:21:51 : 1,500,000 : 0 : 136,449,075 : 2%
2016-01-02 00:30:04 : 1,600,000 : 0 : 159,256,125 : 3%
2016-01-02 01:40:18 : 1,700,000 : 0 : 159,256,125 : 3%
2016-01-02 02:52:26 : 1,800,000 : 0 : 182,194,250 : 3%
2016-01-02 04:05:56 : 1,900,000 : 0 : 

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2016-02.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2016-02_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2016-02.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-02-01 01:00:23 : 100,000 : 0 : 23,200,275 : 0%
2016-02-01 02:00:30 : 200,000 : 0 : 23,200,275 : 0%
2016-02-01 03:03:11 : 300,000 : 0 : 46,400,550 : 1%
2016-02-01 04:09:46 : 400,000 : 0 : 46,400,550 : 1%
2016-02-01 05:23:38 : 500,000 : 0 : 69,338,675 : 1%
2016-02-01 06:55:24 : 600,000 : 0 : 69,338,675 : 1%
2016-02-01 09:00:45 : 700,000 : 0 : 69,338,675 : 1%
2016-02-01 11:33:43 : 800,000 : 0 : 92,670,025 : 2%
2016-02-01 13:27:30 : 900,000 : 0 : 92,670,025 : 2%
2016-02-01 14:45:41 : 1,000,000 : 0 : 116,656,750 : 2%
2016-02-01 15:48:41 : 1,100,000 : 0 : 116,656,750 : 2%
2016-02-01 16:43:04 : 1,200,000 : 0 : 140,643,475 : 2%
2016-02-01 17:35:16 : 1,300,000 : 0 : 140,643,475 : 2%
2016-02-01 18:25:57 : 1,400,000 : 0 : 164,761,275 : 3%
2016-02-01 19:16:20 : 1,500,000 : 0 : 164,761,275 : 3%
2016-02-01 20:06:13 : 1,600,000 : 0 : 164,761,275 : 3%
2016-02-01 20:56:17 : 1,700,000 : 0 : 188,879,075 : 3%
2016-02-01 21:47:17 : 1,800,000 : 0 : 188,879,075 : 3%
2016-02-01 22:39:59 : 1,900,000 : 0 :

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2016-03.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2016-03_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2016-03.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-03-01 01:00:01 : 100,000 : 0 : 23,593,500 : 0%
2016-03-01 01:55:28 : 200,000 : 0 : 23,593,500 : 0%
2016-03-01 02:50:14 : 300,000 : 0 : 46,662,700 : 1%
2016-03-01 03:46:46 : 400,000 : 0 : 46,662,700 : 1%
2016-03-01 04:50:23 : 500,000 : 0 : 69,469,750 : 1%
2016-03-01 06:07:43 : 600,000 : 0 : 69,469,750 : 1%
2016-03-01 07:49:44 : 700,000 : 0 : 69,469,750 : 1%
2016-03-01 10:04:44 : 800,000 : 0 : 92,014,650 : 2%
2016-03-01 12:23:17 : 900,000 : 0 : 92,014,650 : 2%
2016-03-01 13:55:44 : 1,000,000 : 0 : 115,214,925 : 2%
2016-03-01 15:01:02 : 1,100,000 : 0 : 115,214,925 : 2%
2016-03-01 15:55:58 : 1,200,000 : 0 : 138,677,350 : 2%
2016-03-01 16:46:08 : 1,300,000 : 0 : 138,677,350 : 2%
2016-03-01 17:34:11 : 1,400,000 : 0 : 162,270,850 : 3%
2016-03-01 18:22:15 : 1,500,000 : 0 : 162,270,850 : 3%
2016-03-01 19:09:26 : 1,600,000 : 0 : 162,270,850 : 3%
2016-03-01 19:56:58 : 1,700,000 : 0 : 186,126,500 : 3%
2016-03-01 20:43:21 : 1,800,000 : 0 : 186,126,500 : 3%
2016-03-01 21:30:13 : 1,900,000 : 0 :

In [ ]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2016-04.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2016-04_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2016-04.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-04-01 00:56:32 : 100,000 : 0 : 23,462,425 : 0%
2016-04-01 01:54:13 : 200,000 : 0 : 23,462,425 : 0%
2016-04-01 02:52:46 : 300,000 : 0 : 46,269,475 : 1%
2016-04-01 03:56:50 : 400,000 : 0 : 46,269,475 : 1%
2016-04-01 05:02:02 : 500,000 : 0 : 46,269,475 : 1%
2016-04-01 06:30:54 : 600,000 : 0 : 69,338,675 : 1%
2016-04-01 08:28:00 : 700,000 : 0 : 69,338,675 : 1%
2016-04-01 10:43:04 : 800,000 : 0 : 92,670,025 : 2%
2016-04-01 12:26:23 : 900,000 : 0 : 92,670,025 : 2%
2016-04-01 13:37:43 : 1,000,000 : 0 : 92,670,025 : 2%
2016-04-01 14:36:02 : 1,100,000 : 0 : 116,132,450 : 2%
2016-04-01 15:28:29 : 1,200,000 : 0 : 116,132,450 : 2%
2016-04-01 16:19:38 : 1,300,000 : 0 : 139,463,800 : 2%
2016-04-01 17:10:06 : 1,400,000 : 0 : 139,463,800 : 2%
2016-04-01 18:00:55 : 1,500,000 : 0 : 139,463,800 : 2%
2016-04-01 18:51:59 : 1,600,000 : 0 : 162,533,000 : 3%
2016-04-01 19:41:47 : 1,700,000 : 0 : 162,533,000 : 3%
2016-04-01 20:33:00 : 1,800,000 : 0 : 185,733,275 : 3%
2016-04-01 21:28:17 : 1,900,000 : 0 : 

In [6]:
# Define the total file size for percentage calculation.
file_size = os.path.getsize('RC_2016-05.zst')

file_lines = 0
file_bytes_processed = 0
created = None
field = "subreddit"
value = "wallstreetbets"
bad_lines = 0

with open('RC-2016-05_subreddit.json', 'w') as file_written:
  for line, file_bytes_processed in read_lines_zst('RC_2016-05.zst'):
      try:
          obj = json.loads(line)
          created = datetime.utcfromtimestamp(int(obj['created_utc']))
          if obj[field] == value:
              file_written.write(json.dumps(obj) + '\n')
      except (KeyError, json.JSONDecodeError) as err:
          bad_lines += 1
      file_lines += 1
      if file_lines % 100000 == 0:
          print(f"{created.strftime('%Y-%m-%d %H:%M:%S')} : {file_lines:,} : {bad_lines:,} : {file_bytes_processed:,} : {(file_bytes_processed / file_size) * 100:.0f}%")

2016-05-01 01:02:42 : 100,000 : 0 : 23,200,275 : 0%
2016-05-01 02:07:43 : 200,000 : 0 : 23,200,275 : 0%
2016-05-01 03:16:57 : 300,000 : 0 : 46,662,700 : 1%
2016-05-01 04:36:04 : 400,000 : 0 : 46,662,700 : 1%
2016-05-01 06:11:11 : 500,000 : 0 : 46,662,700 : 1%
2016-05-01 08:14:25 : 600,000 : 0 : 70,518,350 : 1%
2016-05-01 10:46:06 : 700,000 : 0 : 70,518,350 : 1%
2016-05-01 12:49:33 : 800,000 : 0 : 93,980,775 : 2%
2016-05-01 14:15:59 : 900,000 : 0 : 93,980,775 : 2%
2016-05-01 15:26:15 : 1,000,000 : 0 : 93,980,775 : 2%
2016-05-01 16:29:24 : 1,100,000 : 0 : 117,705,350 : 2%
2016-05-01 17:27:45 : 1,200,000 : 0 : 117,705,350 : 2%
2016-05-01 18:24:17 : 1,300,000 : 0 : 141,298,850 : 2%
2016-05-01 19:20:30 : 1,400,000 : 0 : 141,298,850 : 2%
2016-05-01 20:16:17 : 1,500,000 : 0 : 141,298,850 : 2%
2016-05-01 21:13:06 : 1,600,000 : 0 : 164,761,275 : 3%
2016-05-01 22:11:51 : 1,700,000 : 0 : 164,761,275 : 3%
2016-05-01 23:13:19 : 1,800,000 : 0 : 187,830,475 : 3%
2016-05-02 00:13:46 : 1,900,000 : 0 : 